In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def sort_by_chromosome(df):
    """Sắp xếp DataFrame theo đúng thứ tự sinh học: 1->22, X, Y, MT và sắp xếp cả POS."""
    print("Đang sắp xếp dữ liệu theo Chromosome và Position...")
    
    # Chuẩn hóa format CHROM (Xóa chữ 'chr' để dễ map)
    df['CHROM'] = df['CHROM'].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    
    # Tạo từ điển quy đổi Chromosome thành số thứ tự
    chrom_order = {str(i): i for i in range(1, 23)}
    chrom_order.update({'X': 23, 'Y': 24, 'MT': 25, 'M': 25})
    
    # Tạo cột tạm để sắp xếp
    df['CHROM_num'] = df['CHROM'].map(chrom_order)
    
    # Những nhiễm sắc thể lạ (alt contigs, unplaced) sẽ bị đẩy xuống cuối cùng
    df['CHROM_num'] = df['CHROM_num'].fillna(99)
    
    # Sort theo CHROM_num, sau đó tới POS (Vị trí) để chuẩn form VCF nhất
    df = df.sort_values(by=['CHROM_num', 'POS'], ascending=[True, True]).reset_index(drop=True)
    
    # Dọn dẹp cột tạm
    df = df.drop(columns=['CHROM_num'])
    
    print("Hoàn tất sắp xếp!")
    return df

def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "--format", "vcf",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals,"
            "SIFT_score,SIFT_converted_rankscore,SIFT_pred,"
            "SIFT4G_score,SIFT4G_converted_rankscore,SIFT4G_pred,"
            "Polyphen2_HDIV_score,Polyphen2_HDIV_rankscore,Polyphen2_HDIV_pred,"
            "Polyphen2_HVAR_score,Polyphen2_HVAR_rankscore,Polyphen2_HVAR_pred,"
            "MutationTaster_score,MutationTaster_rankscore,MutationTaster_pred,"
            "MetaSVM_score,MetaSVM_rankscore,MetaSVM_pred,"
            "MetaLR_score,MetaLR_rankscore,MetaLR_pred,"
            "MetaRNN_score,MetaRNN_rankscore,MetaRNN_pred,"
            "M-CAP_score,M-CAP_rankscore,M-CAP_pred,"
            "REVEL_score,REVEL_rankscore,"
            "MutPred2_score,MutPred2_rankscore,MutPred2_pred,"
            "MVP_score,MVP_rankscore,"
            "gMVP_score,gMVP_rankscore,"
            "MisFit_D_score,MisFit_D_rankscore,MisFit_D_pred_lenient,"
            "MPC_score,MPC_rankscore,"
            "PrimateAI_score,PrimateAI_rankscore,PrimateAI_pred,"
            "BayesDel_addAF_score,BayesDel_addAF_rankscore,BayesDel_addAF_pred,"
            "BayesDel_noAF_score,BayesDel_noAF_rankscore,BayesDel_noAF_pred,"
            "ClinPred_score,ClinPred_rankscore,ClinPred_pred,"
            "LIST-S2_score,LIST-S2_rankscore,LIST-S2_pred,"
            "VARITY_R_score,VARITY_R_rankscore,"
            "VARITY_ER_score,VARITY_ER_rankscore,"
            "AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,"
            "PHACTboost_score,PHACTboost_rankscore,"
            "MutFormer_score,MutFormer_rankscore,"
            "popEVE_score,popEVE_converted_rankscore,popEVE_pred,"
            "CADD_raw,CADD_raw_rankscore,CADD_phred,"
            "DANN_score,DANN_rankscore"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,SYMBOL,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,"
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals,"
            "SIFT_score,SIFT_converted_rankscore,SIFT_pred,"
            "SIFT4G_score,SIFT4G_converted_rankscore,SIFT4G_pred,"
            "Polyphen2_HDIV_score,Polyphen2_HDIV_rankscore,Polyphen2_HDIV_pred,"
            "Polyphen2_HVAR_score,Polyphen2_HVAR_rankscore,Polyphen2_HVAR_pred,"
            "MutationTaster_score,MutationTaster_rankscore,MutationTaster_pred,"
            "MetaSVM_score,MetaSVM_rankscore,MetaSVM_pred,"
            "MetaLR_score,MetaLR_rankscore,MetaLR_pred,"
            "MetaRNN_score,MetaRNN_rankscore,MetaRNN_pred,"
            "M-CAP_score,M-CAP_rankscore,M-CAP_pred,"
            "REVEL_score,REVEL_rankscore,"
            "MutPred2_score,MutPred2_rankscore,MutPred2_pred,"
            "MVP_score,MVP_rankscore,"
            "gMVP_score,gMVP_rankscore,"
            "MisFit_D_score,MisFit_D_rankscore,MisFit_D_pred_lenient,"
            "MPC_score,MPC_rankscore,"
            "PrimateAI_score,PrimateAI_rankscore,PrimateAI_pred,"
            "BayesDel_addAF_score,BayesDel_addAF_rankscore,BayesDel_addAF_pred,"
            "BayesDel_noAF_score,BayesDel_noAF_rankscore,BayesDel_noAF_pred,"
            "ClinPred_score,ClinPred_rankscore,ClinPred_pred,"
            "LIST-S2_score,LIST-S2_rankscore,LIST-S2_pred,"
            "VARITY_R_score,VARITY_R_rankscore,"
            "VARITY_ER_score,VARITY_ER_rankscore,"
            "AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,"
            "PHACTboost_score,PHACTboost_rankscore,"
            "MutFormer_score,MutFormer_rankscore,"
            "popEVE_score,popEVE_converted_rankscore,popEVE_pred,"
            "CADD_raw,CADD_raw_rankscore,CADD_phred,"
            "DANN_score,DANN_rankscore"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols, dtype=str)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    import re
    
    keep_string_cols = [
        'uploaded_variation', 'symbol', 'mane_select', 'canonical', 
        'hgvsc', 'hgvsp', 'ensp', 'consequence', 'amino_acids', 'codons', 
        'feature', 'feature_type', 'cdna_position', 'cds_position', 'protein_position'
    ]
    
    for c in vep_df.columns:
        if c.lower() not in keep_string_cols:
            
            # 1. Nếu là cột Dự đoán (Prediction) - Cần 'gắp' chữ cái (VD: D, T, B)
            if 'pred' in c.lower() and 'score' not in c.lower():
                # Regex tìm chuỗi chữ cái tiếng Anh đầu tiên
                vep_df[c] = vep_df[c].str.extract(r'([a-zA-Z]+)')[0]
            
            # 2. Nếu là cột Điểm số (Score, Rankscore, AF, GERP, phyloP...)
            else:
                # Regex siêu việt: Bắt số nguyên, số thập phân, và cả số mũ (VD: 1.5e-5)
                # Nó sẽ tự động ngó lơ các dấu phẩy (,), dấu chấm lẻ (.) và dấu (&)
                vep_df[c] = vep_df[c].str.extract(r'([-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?)')[0]
                vep_df[c] = pd.to_numeric(vep_df[c], errors='coerce')
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")
    
    # ---------------------------------------------------------
    # TẠO NHÃN PREDICTION (D/T) CHO CÁC MODEL KHÔNG CÓ SẴN CỘT PRED
    # ---------------------------------------------------------
    
    # Hàm hỗ trợ gắn nhãn an toàn (Bỏ qua NaN)
    def assign_pred(score, threshold):
        if pd.isna(score):
            return None
        return 'D' if score >= threshold else 'T'

    # BỔ SUNG ĐẦY ĐỦ TẤT CẢ CÁC CỘT SCORE VÀO DANH SÁCH NÀY
    custom_pred_models = [
        'PHACTboost_score', 'MutFormer_score', 'MVP_score', 'REVEL_score', 
        'CADD_phred', 'DANN_score', 'gMVP_score', 'MPC_score', 
        'VARITY_R_score', 'VARITY_ER_score'
    ]
    
    # Ép toàn bộ các cột này về kiểu số thực (Float), biến '-' thành NaN
    for col in custom_pred_models:
        if col in merged_chunk.columns:
            merged_chunk[col] = pd.to_numeric(merged_chunk[col], errors='coerce')

    # 1. PHACTboost (Ngưỡng README: 0.62)
    if 'PHACTboost_score' in merged_chunk.columns:
        merged_chunk['PHACTboost_pred'] = merged_chunk['PHACTboost_score'].apply(lambda x: assign_pred(x, 0.62))

    # 2. MutFormer (Ngưỡng README: 0.8838)
    if 'MutFormer_score' in merged_chunk.columns:
        merged_chunk['MutFormer_pred'] = merged_chunk['MutFormer_score'].apply(lambda x: assign_pred(x, 0.8838))

    # 3. MVP (Ngưỡng README: 0.75)
    if 'MVP_score' in merged_chunk.columns:
        merged_chunk['MVP_pred'] = merged_chunk['MVP_score'].apply(lambda x: assign_pred(x, 0.75))

    # 4. REVEL (Tiêu chuẩn cộng đồng lâm sàng: 0.5)
    if 'REVEL_score' in merged_chunk.columns:
        merged_chunk['REVEL_pred'] = merged_chunk['REVEL_score'].apply(lambda x: assign_pred(x, 0.5))

    # 5. CADD Phred (Tiêu chuẩn vàng: 20)
    if 'CADD_phred' in merged_chunk.columns:
        merged_chunk['CADD_pred'] = merged_chunk['CADD_phred'].apply(lambda x: assign_pred(x, 20.0))

    # 6. DANN (Tiêu chuẩn cộng đồng: 0.9)
    if 'DANN_score' in merged_chunk.columns:
        merged_chunk['DANN_pred'] = merged_chunk['DANN_score'].apply(lambda x: assign_pred(x, 0.9))

    # 7. gMVP (Luật ngầm: 0.5)
    if 'gMVP_score' in merged_chunk.columns:
        merged_chunk['gMVP_pred'] = merged_chunk['gMVP_score'].apply(lambda x: assign_pred(x, 0.5))

    # 8. MPC (Luật ngầm lâm sàng: >= 2.0)
    if 'MPC_score' in merged_chunk.columns:
        merged_chunk['MPC_pred'] = merged_chunk['MPC_score'].apply(lambda x: assign_pred(x, 2.0))

    # 9. VARITY_R & VARITY_ER (Luật ngầm: 0.5)
    if 'VARITY_R_score' in merged_chunk.columns:
        merged_chunk['VARITY_R_pred'] = merged_chunk['VARITY_R_score'].apply(lambda x: assign_pred(x, 0.5))
        
    if 'VARITY_ER_score' in merged_chunk.columns:
        merged_chunk['VARITY_ER_pred'] = merged_chunk['VARITY_ER_score'].apply(lambda x: assign_pred(x, 0.5))

    for col in merged_chunk.columns:
        if merged_chunk[col].dtype == 'object':
            merged_chunk[col] = merged_chunk[col].astype('string')

    return merged_chunk

def process_chunk_for_test_clinvar_dbsnp_merged(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "--format", "vcf",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals,"
            "SIFT_score,SIFT_converted_rankscore,SIFT_pred,"
            "SIFT4G_score,SIFT4G_converted_rankscore,SIFT4G_pred,"
            "Polyphen2_HDIV_score,Polyphen2_HDIV_rankscore,Polyphen2_HDIV_pred,"
            "Polyphen2_HVAR_score,Polyphen2_HVAR_rankscore,Polyphen2_HVAR_pred,"
            "MutationTaster_score,MutationTaster_rankscore,MutationTaster_pred,"
            "MetaSVM_score,MetaSVM_rankscore,MetaSVM_pred,"
            "MetaLR_score,MetaLR_rankscore,MetaLR_pred,"
            "MetaRNN_score,MetaRNN_rankscore,MetaRNN_pred,"
            "M-CAP_score,M-CAP_rankscore,M-CAP_pred,"
            "REVEL_score,REVEL_rankscore,"
            "MutPred2_score,MutPred2_rankscore,MutPred2_pred,"
            "MVP_score,MVP_rankscore,"
            "gMVP_score,gMVP_rankscore,"
            "MisFit_D_score,MisFit_D_rankscore,MisFit_D_pred_lenient,"
            "MPC_score,MPC_rankscore,"
            "PrimateAI_score,PrimateAI_rankscore,PrimateAI_pred,"
            "BayesDel_addAF_score,BayesDel_addAF_rankscore,BayesDel_addAF_pred,"
            "BayesDel_noAF_score,BayesDel_noAF_rankscore,BayesDel_noAF_pred,"
            "ClinPred_score,ClinPred_rankscore,ClinPred_pred,"
            "LIST-S2_score,LIST-S2_rankscore,LIST-S2_pred,"
            "VARITY_R_score,VARITY_R_rankscore,"
            "VARITY_ER_score,VARITY_ER_rankscore,"
            "AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,"
            "PHACTboost_score,PHACTboost_rankscore,"
            "MutFormer_score,MutFormer_rankscore,"
            "popEVE_score,popEVE_converted_rankscore,popEVE_pred,"
            "CADD_raw,CADD_raw_rankscore,CADD_phred,"
            "DANN_score,DANN_rankscore"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,"
            "SIFT_score,SIFT_converted_rankscore,SIFT_pred,"
            "SIFT4G_score,SIFT4G_converted_rankscore,SIFT4G_pred,"
            "Polyphen2_HDIV_score,Polyphen2_HDIV_rankscore,Polyphen2_HDIV_pred,"
            "Polyphen2_HVAR_score,Polyphen2_HVAR_rankscore,Polyphen2_HVAR_pred,"
            "MutationTaster_score,MutationTaster_rankscore,MutationTaster_pred,"
            "MetaSVM_score,MetaSVM_rankscore,MetaSVM_pred,"
            "MetaLR_score,MetaLR_rankscore,MetaLR_pred,"
            "MetaRNN_score,MetaRNN_rankscore,MetaRNN_pred,"
            "M-CAP_score,M-CAP_rankscore,M-CAP_pred,"
            "REVEL_score,REVEL_rankscore,"
            "MutPred2_score,MutPred2_rankscore,MutPred2_pred,"
            "MVP_score,MVP_rankscore,"
            "gMVP_score,gMVP_rankscore,"
            "MisFit_D_score,MisFit_D_rankscore,MisFit_D_pred_lenient,"
            "MPC_score,MPC_rankscore,"
            "PrimateAI_score,PrimateAI_rankscore,PrimateAI_pred,"
            "BayesDel_addAF_score,BayesDel_addAF_rankscore,BayesDel_addAF_pred,"
            "BayesDel_noAF_score,BayesDel_noAF_rankscore,BayesDel_noAF_pred,"
            "ClinPred_score,ClinPred_rankscore,ClinPred_pred,"
            "LIST-S2_score,LIST-S2_rankscore,LIST-S2_pred,"
            "VARITY_R_score,VARITY_R_rankscore,"
            "VARITY_ER_score,VARITY_ER_rankscore,"
            "AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,"
            "PHACTboost_score,PHACTboost_rankscore,"
            "MutFormer_score,MutFormer_rankscore,"
            "popEVE_score,popEVE_converted_rankscore,popEVE_pred,"
            "CADD_raw,CADD_raw_rankscore,CADD_phred,"
            "DANN_score,DANN_rankscore"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        result = subprocess.run(
            vep_cmd,
            check=True,
            capture_output=True,
            text=True
        )
        print(f"[VEP CHUNK {chunk_idx} THÀNH CÔNG]")

    except subprocess.CalledProcessError as e:
        print("\n" + "=" * 70)
        print(f"[VEP ERROR - CHUNK {chunk_idx}]")
        print("=" * 70)
        print("\nReturn code:")
        print(e.returncode)
        print("\nSTDERR:")
        print(e.stderr if e.stderr else "<empty>")
        raise SystemExit(f"Dừng pipeline do VEP failed at chunk {chunk_idx}")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    # vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols, dtype=str)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    import re
    
    keep_string_cols = [
            'uploaded_variation', 'symbol', 'mane_select', 'canonical', 
            'hgvsc', 'hgvsp', 'ensp', 'consequence', 'amino_acids', 'codons', 
            'feature', 'feature_type', 'cdna_position', 'cds_position', 'protein_position'
        ]
        
    for c in vep_df.columns:
        if c.lower() not in keep_string_cols:
            
            # 1. Nếu là cột Dự đoán (Prediction) - Cần 'gắp' chữ cái (VD: D, T, B)
            if 'pred' in c.lower() and 'score' not in c.lower():
                # Regex tìm chuỗi chữ cái tiếng Anh đầu tiên
                vep_df[c] = vep_df[c].str.extract(r'([a-zA-Z]+)')[0]
            
            # 2. Nếu là cột Điểm số (Score, Rankscore, AF, GERP, phyloP...)
            else:
                # Regex siêu việt: Bắt số nguyên, số thập phân, và cả số mũ (VD: 1.5e-5)
                # Nó sẽ tự động ngó lơ các dấu phẩy (,), dấu chấm lẻ (.) và dấu (&)
                vep_df[c] = vep_df[c].str.extract(r'([-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?)')[0]
                vep_df[c] = pd.to_numeric(vep_df[c], errors='coerce')
    
    # Merge lại với chunk ban đầu
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")
    
    # ---------------------------------------------------------
    # TẠO NHÃN PREDICTION (D/T) CHO CÁC MODEL KHÔNG CÓ SẴN CỘT PRED
    # ---------------------------------------------------------
    
    # Hàm hỗ trợ gắn nhãn an toàn (Bỏ qua NaN)
    def assign_pred(score, threshold):
        if pd.isna(score):
            return None
        return 'D' if score >= threshold else 'T'

    # BỔ SUNG ĐẦY ĐỦ TẤT CẢ CÁC CỘT SCORE VÀO DANH SÁCH NÀY
    custom_pred_models = [
        'PHACTboost_score', 'MutFormer_score', 'MVP_score', 'REVEL_score', 
        'CADD_phred', 'DANN_score', 'gMVP_score', 'MPC_score', 
        'VARITY_R_score', 'VARITY_ER_score'
    ]
    
    # Ép toàn bộ các cột này về kiểu số thực (Float), biến '-' thành NaN
    for col in custom_pred_models:
        if col in merged_chunk.columns:
            merged_chunk[col] = pd.to_numeric(merged_chunk[col], errors='coerce')

    # 1. PHACTboost (Ngưỡng README: 0.62)
    if 'PHACTboost_score' in merged_chunk.columns:
        merged_chunk['PHACTboost_pred'] = merged_chunk['PHACTboost_score'].apply(lambda x: assign_pred(x, 0.62))

    # 2. MutFormer (Ngưỡng README: 0.8838)
    if 'MutFormer_score' in merged_chunk.columns:
        merged_chunk['MutFormer_pred'] = merged_chunk['MutFormer_score'].apply(lambda x: assign_pred(x, 0.8838))

    # 3. MVP (Ngưỡng README: 0.75)
    if 'MVP_score' in merged_chunk.columns:
        merged_chunk['MVP_pred'] = merged_chunk['MVP_score'].apply(lambda x: assign_pred(x, 0.75))

    # 4. REVEL (Tiêu chuẩn cộng đồng lâm sàng: 0.5)
    if 'REVEL_score' in merged_chunk.columns:
        merged_chunk['REVEL_pred'] = merged_chunk['REVEL_score'].apply(lambda x: assign_pred(x, 0.5))

    # 5. CADD Phred (Tiêu chuẩn vàng: 20)
    if 'CADD_phred' in merged_chunk.columns:
        merged_chunk['CADD_pred'] = merged_chunk['CADD_phred'].apply(lambda x: assign_pred(x, 20.0))

    # 6. DANN (Tiêu chuẩn cộng đồng: 0.9)
    if 'DANN_score' in merged_chunk.columns:
        merged_chunk['DANN_pred'] = merged_chunk['DANN_score'].apply(lambda x: assign_pred(x, 0.9))

    # 7. gMVP (Luật ngầm: 0.5)
    if 'gMVP_score' in merged_chunk.columns:
        merged_chunk['gMVP_pred'] = merged_chunk['gMVP_score'].apply(lambda x: assign_pred(x, 0.5))

    # 8. MPC (Luật ngầm lâm sàng: >= 2.0)
    if 'MPC_score' in merged_chunk.columns:
        merged_chunk['MPC_pred'] = merged_chunk['MPC_score'].apply(lambda x: assign_pred(x, 2.0))

    # 9. VARITY_R & VARITY_ER (Luật ngầm: 0.5)
    if 'VARITY_R_score' in merged_chunk.columns:
        merged_chunk['VARITY_R_pred'] = merged_chunk['VARITY_R_score'].apply(lambda x: assign_pred(x, 0.5))
        
    if 'VARITY_ER_score' in merged_chunk.columns:
        merged_chunk['VARITY_ER_pred'] = merged_chunk['VARITY_ER_score'].apply(lambda x: assign_pred(x, 0.5))

    for col in merged_chunk.columns:
        if merged_chunk[col].dtype == 'object':
            merged_chunk[col] = merged_chunk[col].astype('string')
    
    return merged_chunk

In [2]:
INPUT_PARQUET = r"D:/variant_data/test.parquet"
FINAL_OUTPUT_PARQUET = r"D:/variant_data/test_after_vep.parquet"

print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
df_full = sort_by_chromosome(df_full)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk_for_test_clinvar_dbsnp_merged(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Đang tải dữ liệu từ D:/variant_data/test.parquet...
Đang sắp xếp dữ liệu theo Chromosome và Position...
Hoàn tất sắp xếp!
[*] Tổng số variant: 11,864. Số lượng chunks: 1
[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 11864)...
[VEP CHUNK 1 THÀNH CÔNG]

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/test_after_vep.parquet


In [3]:
INPUT_PARQUET = r"D:/variant_data/clinvarhq_ready_for_vep.parquet"
FINAL_OUTPUT_PARQUET = r"D:/variant_data/clinvarhq_after_vep.parquet"

print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Đang tải dữ liệu từ D:/variant_data/clinvarhq_ready_for_vep.parquet...
[*] Tổng số variant: 3,661. Số lượng chunks: 1
[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 3661)...

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/clinvarhq_after_vep.parquet


In [4]:
INPUT_PARQUET = r"D:/variant_data/proteingym_ready_for_vep.parquet"
FINAL_OUTPUT_PARQUET = r"D:/variant_data/proteingym_after_vep.parquet"

print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Đang tải dữ liệu từ D:/variant_data/proteingym_ready_for_vep.parquet...
[*] Tổng số variant: 4,490. Số lượng chunks: 1
[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 4490)...

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/proteingym_after_vep.parquet


In [5]:
INPUT_PARQUET = r"D:/variant_data/uniprot_ready_for_vep.parquet"
FINAL_OUTPUT_PARQUET = r"D:/variant_data/uniprot_after_vep.parquet"

print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Đang tải dữ liệu từ D:/variant_data/uniprot_ready_for_vep.parquet...
[*] Tổng số variant: 5,742. Số lượng chunks: 1
[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 5742)...

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/uniprot_after_vep.parquet


In [25]:
import numpy as np
import pandas as pd

final_df = pd.read_parquet(r"D:\variant_data\test_after_vep.parquet")

# Danh sách giá trị cần kiểm tra
invalid_values = ['.', "-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)

print(final_df['Consequence'].value_counts(),'\n')
final_df = final_df[final_df['Consequence'] == 'missense_variant']

print(final_df.isnull().sum()[60:120])
print(final_df.isnull().sum()[120:])

final_df

Consequence
missense_variant    11864
Name: count, dtype: int64 

clnrevstat                    10061
clnacc                        10061
Source                            0
ENSP                              0
HGVSc                             0
HGVSp                             0
Review_Rank                       0
Split                             0
SIFT_score                      496
SIFT_converted_rankscore        496
SIFT_pred                       496
SIFT4G_score                    671
SIFT4G_converted_rankscore      671
SIFT4G_pred                     671
Polyphen2_HDIV_score            706
Polyphen2_HDIV_rankscore        706
Polyphen2_HDIV_pred             706
Polyphen2_HVAR_score            706
Polyphen2_HVAR_rankscore        706
Polyphen2_HVAR_pred             706
MutationTaster_score            691
MutationTaster_rankscore        691
MutationTaster_pred             691
MetaSVM_score                    99
MetaSVM_rankscore                99
MetaSVM_pred                     9

,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
0,10_47121_T_C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,T,D,NaN,T,T,D,NaN,T,T
1,10_48834_G_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,T,T,NaN,T,T,D,NaN,D,T
2,10_49229_T_G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,T,T,D,NaN,T,T,D,NaN,T,T
3,10_49234_C_T,222975.0,single nucleotide variant,NM_177987.3(TUBB8):c.5G>A (p.Arg2Lys),347688.0,TUBB8,HGNC:20773,Pathogenic,1.0,"MONDO:MONDO:0021573,MedGen:C4225210,OMIM:616780",...,D,T,D,NaN,D,T,D,NaN,T,T
4,10_180088_C_T,424656.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.76C>T (p.Arg26Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0014486,MedGen:C4015167,OMIM:61608...",...,D,T,D,T,D,D,D,D,T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11859,20_64208272_G_A,2367599.0,single nucleotide variant,NM_004535.3(MYT1):c.1076G>A (p.Arg359Gln),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,T,T,T,T,T
11860,20_64208481_A_G,742556.0,single nucleotide variant,NM_004535.3(MYT1):c.1285A>G (p.Ser429Gly),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900,...,T,T,T,T,D,T,T,T,T,T
11861,20_64217278_C_A,3447695.0,single nucleotide variant,NM_004535.3(MYT1):c.1843C>A (p.Gln615Lys),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,D,D,T,T,T
11862,20_64221996_C_G,3191571.0,single nucleotide variant,NM_004535.3(MYT1):c.2345C>G (p.Thr782Ser),4661.0,MYT1,HGNC:7622,Benign,0.0,NaN,...,T,T,NaN,T,T,T,T,T,T,T


In [26]:
# Giả sử df là DataFrame hiện tại của bạn
print(f"🔹 Số lượng biến thể ban đầu: {len(final_df):,}")

# 1. Tự động nhận diện tất cả các cột điểm số 
# (Bao gồm: score, rankscore, phred, raw, và các điểm bảo tồn tiến hóa như gerp, phylop, phastcons)
score_cols = [
    col for col in final_df.columns 
    if any(keyword in col.lower() for keyword in ['score', 'phred', 'raw'])
]

print(f"🔹 Đã tìm thấy {len(score_cols)} cột điểm số để kiểm tra.")

# 2. Xóa các dòng bị thiếu (NaN) ở BẤT KỲ cột điểm số nào trong danh sách trên
df_clean = final_df.dropna(subset=score_cols).copy()

# 3. Báo cáo kết quả
dropped_count = len(final_df) - len(df_clean)
print(f"🔹 Số lượng biến thể sau khi xóa: {len(df_clean):,}")
print(f"⚠️ Đã xóa tổng cộng: {dropped_count:,} dòng (chiếm {(dropped_count/len(final_df)*100):.2f}%)")

print(df_clean.isnull().sum()[60:120])
print(df_clean.isnull().sum()[120:])

df_clean.to_parquet(r"D:\variant_data\test_after_vep_final.parquet", index=False)

df_clean

🔹 Số lượng biến thể ban đầu: 11,864
🔹 Đã tìm thấy 57 cột điểm số để kiểm tra.
🔹 Số lượng biến thể sau khi xóa: 6,095
⚠️ Đã xóa tổng cộng: 5,769 dòng (chiếm 48.63%)
clnrevstat                    5135
clnacc                        5135
Source                           0
ENSP                             0
HGVSc                            0
HGVSp                            0
Review_Rank                      0
Split                            0
SIFT_score                       0
SIFT_converted_rankscore         0
SIFT_pred                        0
SIFT4G_score                     0
SIFT4G_converted_rankscore       0
SIFT4G_pred                      0
Polyphen2_HDIV_score             0
Polyphen2_HDIV_rankscore         0
Polyphen2_HDIV_pred              0
Polyphen2_HVAR_score             0
Polyphen2_HVAR_rankscore         0
Polyphen2_HVAR_pred              0
MutationTaster_score             0
MutationTaster_rankscore         0
MutationTaster_pred              0
MetaSVM_score                  

,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
4,10_180088_C_T,424656.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.76C>T (p.Arg26Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0014486,MedGen:C4015167,OMIM:61608...",...,D,T,D,T,D,D,D,D,T,T
5,10_237638_G_T,2038412.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.570G>T (p.Arg190Ser),10771.0,ZMYND11,HGNC:16966,Likely benign,0.0,"MedGen:C3661900|MeSH:D030342,MedGen:C0950123",...,D,T,T,T,T,D,D,T,D,T
6,10_240913_C_G,2077177.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.774C>G (p.Cys258Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic,1.0,MedGen:C3661900,...,D,T,D,T,D,D,D,D,D,T
7,10_242031_A_C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,T,D,D,D,D,D,D,D,D
8,10_246823_C_G,3002871.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.1008C>G (p.His336Gln),10771.0,ZMYND11,HGNC:16966,Benign,0.0,MedGen:C3661900,...,T,T,T,T,T,T,D,T,D,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11856,20_64208061_G_A,728821.0,single nucleotide variant,NM_004535.3(MYT1):c.865G>A (p.Glu289Lys),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900|,...,T,T,T,T,D,T,T,T,T,T
11857,20_64208182_G_A,4228399.0,single nucleotide variant,NM_004535.3(MYT1):c.986G>A (p.Arg329Gln),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,T,T,T,T,T
11859,20_64208272_G_A,2367599.0,single nucleotide variant,NM_004535.3(MYT1):c.1076G>A (p.Arg359Gln),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,T,T,T,T,T
11860,20_64208481_A_G,742556.0,single nucleotide variant,NM_004535.3(MYT1):c.1285A>G (p.Ser429Gly),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900,...,T,T,T,T,D,T,T,T,T,T


In [32]:
import numpy as np
import pandas as pd

final_df = pd.read_parquet(r"D:\variant_data\uniprot_after_vep.parquet")

# Danh sách giá trị cần kiểm tra
invalid_values = ['.', "-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)

print(final_df['Consequence'].value_counts(),'\n')
final_df = final_df[final_df['Consequence'] == 'missense_variant']

print(final_df.isnull().sum()[10:70])
print(final_df.isnull().sum()[70:])

final_df

Consequence
missense_variant                                            5143
missense_variant,splice_region_variant                       158
downstream_gene_variant                                      133
synonymous_variant                                           108
upstream_gene_variant                                         81
stop_gained                                                   56
intron_variant                                                20
non_coding_transcript_exon_variant                            16
intron_variant,non_coding_transcript_variant                   7
splice_region_variant,synonymous_variant                       6
3_prime_UTR_variant                                            6
stop_gained,splice_region_variant                              3
5_prime_UTR_variant                                            2
splice_region_variant,non_coding_transcript_exon_variant       1
start_lost                                                     1
intergenic_va

,Variant_ID,CHROM,POS,REF,ALT,Label,dbSNP,gene,protein_AC,aa_change,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
0,10_47143_C_T,10,47143,C,T,1,rs869025272,TUBB8,Q3ZCM7,p.Asp417Asn,...,D,T,D,NaN,T,D,D,NaN,T,T
1,10_47304_A_G,10,47304,A,G,1,rs869025611,TUBB8,Q3ZCM7,p.Met363Thr,...,D,T,D,NaN,T,T,D,NaN,D,D
2,10_47349_T_C,10,47349,T,C,1,rs1270068662,TUBB8,Q3ZCM7,p.Asn348Ser,...,D,T,D,NaN,T,T,D,NaN,D,T
3,10_47359_G_A,10,47359,G,A,0,rs4880608,TUBB8,Q3ZCM7,p.Leu345Phe,...,T,T,T,NaN,T,T,D,NaN,D,T
4,10_47433_C_T,10,47433,C,T,1,rs1465781298,TUBB8,Q3ZCM7,p.Arg320His,...,D,T,T,NaN,T,T,D,NaN,T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5736,Y_12833779_G_T,Y,12833779,G,T,0,rs2032606,USP9Y,O00507,p.Ala1705Ser,...,T,T,T,NaN,D,D,T,T,D,T
5737,Y_19707590_C_G,Y,19707590,C,G,0,rs1050807,KDM5D,Q9BY66,p.Val1186Leu,...,T,T,T,T,T,T,T,NaN,T,T
5739,Y_57190014_G_A,Y,57190014,G,A,0,rs3093495,IL9R,Q01113,p.Arg63Lys,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5740,Y_57191933_G_C,Y,57191933,G,C,0,rs6522,IL9R,Q01113,p.Glu239Gln,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# Giả sử df là DataFrame hiện tại của bạn
print(f"🔹 Số lượng biến thể ban đầu: {len(final_df):,}")

# 1. Tự động nhận diện tất cả các cột điểm số 
# (Bao gồm: score, rankscore, phred, raw, và các điểm bảo tồn tiến hóa như gerp, phylop, phastcons)
score_cols = [
    col for col in final_df.columns 
    if any(keyword in col.lower() for keyword in ['score', 'phred', 'raw', 'gerp', 'phylop', 'phastcons'])
]

print(f"🔹 Đã tìm thấy {len(score_cols)} cột điểm số để kiểm tra.")

# 2. Xóa các dòng bị thiếu (NaN) ở BẤT KỲ cột điểm số nào trong danh sách trên
df_clean = final_df.dropna(subset=score_cols).copy()
df_clean[['AF', 'gnomADe_AF']] = df_clean[['AF', 'gnomADe_AF']].fillna(0)

# 3. Báo cáo kết quả
dropped_count = len(final_df) - len(df_clean)
print(f"🔹 Số lượng biến thể sau khi xóa: {len(df_clean):,}")
print(f"⚠️ Đã xóa tổng cộng: {dropped_count:,} dòng (chiếm {(dropped_count/len(final_df)*100):.2f}%)")

print(df_clean.isnull().sum()[10:70])
print(df_clean.isnull().sum()[70:])

df_clean.to_parquet(r"D:\variant_data\uniprot_after_vep_final.parquet", index=False)

df_clean

🔹 Số lượng biến thể ban đầu: 5,143
🔹 Đã tìm thấy 66 cột điểm số để kiểm tra.
🔹 Số lượng biến thể sau khi xóa: 1,333
⚠️ Đã xóa tổng cộng: 3,810 dòng (chiếm 74.08%)
disease                        669
Location                         0
Allele                        1333
Gene                             0
SYMBOL                           0
Feature                          0
Feature_type                     0
Consequence                      0
cDNA_position                    0
CDS_position                     0
Protein_position                 0
Amino_acids                      0
Codons                           0
MANE_SELECT                      0
CANONICAL                        0
AF                               0
gnomADe_AF                       0
phyloP100way_vertebrate          0
phyloP470way_mammalian           0
phyloP17way_primate              0
phastCons100way_vertebrate       0
phastCons470way_mammalian        0
phastCons17way_primate           0
GERP++_RS                       

,Variant_ID,CHROM,POS,REF,ALT,Label,dbSNP,gene,protein_AC,aa_change,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
16,10_1072186_G_A,10,1072186,G,A,0,rs17856557,WDR37,Q9Y2I8,p.Ala11Thr,...,T,T,T,T,D,D,D,T,T,T
18,10_1096193_A_G,10,1096193,A,G,0,rs2306407,WDR37,Q9Y2I8,p.Ile225Val,...,T,T,T,T,T,T,T,T,T,T
20,10_1379131_C_A,10,1379131,C,A,0,rs3793733,ADARB2,Q9NS39,p.Ala44Thr,...,T,T,T,T,T,D,T,T,T,T
28,10_3151324_G_A,10,3151324,G,A,0,rs12248937,PITRM1,Q5JRX3,p.Ala554Asp,...,T,T,T,T,T,T,T,T,D,T
32,10_3165320_C_T,10,3165320,C,T,1,rs1249144069,PITRM1,Q5JRX3,p.Arg183Gln,...,D,T,D,T,D,D,D,T,D,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5616,20_63495972_C_T,20,63495972,C,T,1,rs587777162,EEF1A2,Q05639,p.Gly70Ser,...,D,D,D,D,D,D,D,D,D,D
5629,20_63547241_C_A,20,63547241,C,A,0,rs55863722,SRMS,Q9H3Y6,p.Gly75Arg,...,D,T,D,D,D,D,D,T,D,D
5630,20_63547247_G_A,20,63547247,G,A,0,rs56053583,SRMS,Q9H3Y6,p.Arg73Cys,...,T,T,T,T,T,D,T,T,T,T
5660,20_63708763_C_A,20,63708763,C,A,0,rs1291212,ZGPAT,Q8N5A5,p.Ser61Arg,...,T,T,T,T,T,D,T,T,T,T


In [34]:
import numpy as np
import pandas as pd

final_df = pd.read_parquet(r"D:\variant_data\clinvarhq_after_vep.parquet")

# Danh sách giá trị cần kiểm tra
invalid_values = ['.', "-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)

print(final_df['Consequence'].value_counts(),'\n')
final_df = final_df[final_df['Consequence'] == 'missense_variant']

print(final_df.isnull().sum()[10:70])
print(final_df.isnull().sum()[70:])

final_df

Consequence
missense_variant                                      3279
missense_variant,splice_region_variant                 125
downstream_gene_variant                                 71
intron_variant                                          51
upstream_gene_variant                                   44
synonymous_variant                                      31
start_lost                                              29
3_prime_UTR_variant                                     10
splice_donor_variant                                     5
5_prime_UTR_variant                                      5
intron_variant,non_coding_transcript_variant             2
splice_region_variant,synonymous_variant                 2
stop_gained                                              2
splice_region_variant,3_prime_UTR_variant                1
splice_acceptor_variant                                  1
splice_polypyrimidine_tract_variant,intron_variant       1
splice_donor_region_variant,intron_variant  

,Variant_ID,CHROM,POS,REF,ALT,Label,ID,GeneInfo,CLNSIG,CLNREVSTAT,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
0,10_47629_C_T,10,47629,C,T,1,694522,TUBB8,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,D,NaN,T,D,D,NaN,D,T
1,10_180020_G_A,10,180020,G,A,0,2907981,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,T,D,D,D,D,T,T
2,10_237638_G_T,10,237638,G,T,0,1980541,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,T,T,D,D,T,D,T
3,10_240949_C_A,10,240949,C,A,0,2074880,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,D,D,T,T,T
4,10_242115_G_A,10,242115,G,A,1,1064586,ZMYND11,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,D,D,D,D,D,D,D,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3656,20_63873800_G_A,20,63873800,G,A,0,790891,TPD52L2,Benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,D,D,T,T,T
3657,20_63930873_T_G,20,63930873,T,G,1,30894,DNAJC5,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,D,D,D,D,D,T,D,D
3658,20_63946593_C_T,20,63946593,C,T,0,2591760,UCKL1,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,D,T,T,T,T
3659,20_64048912_C_T,20,64048912,C,T,0,709170,SOX18,Benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,D,D,D,D,NaN,D,D


In [35]:
# Giả sử df là DataFrame hiện tại của bạn
print(f"🔹 Số lượng biến thể ban đầu: {len(final_df):,}")

# 1. Tự động nhận diện tất cả các cột điểm số 
# (Bao gồm: score, rankscore, phred, raw, và các điểm bảo tồn tiến hóa như gerp, phylop, phastcons)
score_cols = [
    col for col in final_df.columns 
    if any(keyword in col.lower() for keyword in ['score', 'phred', 'raw', 'gerp', 'phylop', 'phastcons'])
]

print(f"🔹 Đã tìm thấy {len(score_cols)} cột điểm số để kiểm tra.")

# 2. Xóa các dòng bị thiếu (NaN) ở BẤT KỲ cột điểm số nào trong danh sách trên
df_clean = final_df.dropna(subset=score_cols).copy()
df_clean[['AF', 'gnomADe_AF']] = df_clean[['AF', 'gnomADe_AF']].fillna(0)

# 3. Báo cáo kết quả
dropped_count = len(final_df) - len(df_clean)
print(f"🔹 Số lượng biến thể sau khi xóa: {len(df_clean):,}")
print(f"⚠️ Đã xóa tổng cộng: {dropped_count:,} dòng (chiếm {(dropped_count/len(final_df)*100):.2f}%)")

print(df_clean.isnull().sum()[10:70])
print(df_clean.isnull().sum()[70:])

df_clean.to_parquet(r"D:\variant_data\clinvarhq_after_vep_final.parquet", index=False)

df_clean

🔹 Số lượng biến thể ban đầu: 3,279
🔹 Đã tìm thấy 66 cột điểm số để kiểm tra.
🔹 Số lượng biến thể sau khi xóa: 703
⚠️ Đã xóa tổng cộng: 2,576 dòng (chiếm 78.56%)
Location                        0
Allele                        703
Gene                            0
SYMBOL                          0
Feature                         0
Feature_type                    0
Consequence                     0
cDNA_position                   0
CDS_position                    0
Protein_position                0
Amino_acids                     0
Codons                          0
MANE_SELECT                     0
CANONICAL                       0
AF                              0
gnomADe_AF                      0
phyloP100way_vertebrate         0
phyloP470way_mammalian          0
phyloP17way_primate             0
phastCons100way_vertebrate      0
phastCons470way_mammalian       0
phastCons17way_primate          0
GERP++_RS                       0
GERP++_NR                       0
GERP_92_mammals        

,Variant_ID,CHROM,POS,REF,ALT,Label,ID,GeneInfo,CLNSIG,CLNREVSTAT,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
2,10_237638_G_T,10,237638,G,T,0,1980541,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,T,T,D,D,T,D,T
6,10_357852_G_C,10,357852,G,C,0,2758294,DIP2C,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,T,T,T,T,T
8,10_1086292_C_T,10,1086292,C,T,0,2279214,WDR37,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,D,D,D,D,T,D,D
38,10_5102114_C_T,10,5102114,C,T,0,716665,AKR1C3,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,T,D,T,D,T
53,10_8064041_G_A,10,8064041,G,A,1,3384342,GATA3,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,D,D,D,D,D,T,D,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3640,20_63488381_C_A,20,63488381,C,A,1,383531,EEF1A2,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,T,T,D,D,D,D,D,D
3641,20_63495909_C_T,20,63495909,C,T,1,279803,EEF1A2,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,D,D,D,D,D,D,D,D
3642,20_63495972_C_T,20,63495972,C,T,1,100782,EEF1A2,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,D,D,D,D,D,D,D,D
3657,20_63930873_T_G,20,63930873,T,G,1,30894,DNAJC5,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,T,D,D,D,D,D,T,D,D


In [36]:
import numpy as np
import pandas as pd

final_df = pd.read_parquet(r"D:\variant_data\proteingym_after_vep.parquet")

# Danh sách giá trị cần kiểm tra
invalid_values = ['.', "-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)

print(final_df['Consequence'].value_counts(),'\n')
final_df = final_df[final_df['Consequence'] == 'missense_variant']

print(final_df.isnull().sum()[10:70])
print(final_df.isnull().sum()[70:])

final_df

Consequence
missense_variant                                                            3717
synonymous_variant                                                           408
missense_variant,splice_region_variant                                       108
stop_gained                                                                   88
downstream_gene_variant                                                       74
upstream_gene_variant                                                         42
splice_region_variant,synonymous_variant                                      16
intron_variant                                                                14
3_prime_UTR_variant                                                            7
5_prime_UTR_variant                                                            6
stop_gained,splice_region_variant                                              4
start_lost                                                                     3
splice_region_va

,Variant_ID,CHROM,POS,REF,ALT,Label,protein,protein_sequence,mutant,mutated_sequence,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
0,10_180016_G_A,10,180016,G,A,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,A2T,MTRLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,T,T,D,D,D,T,T,T
1,10_180088_C_T,10,180088,C,T,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R26W,MARLTKRRQADTKAIQHLWAAIEIIWNQKQIANIDRITKYMSRVHG...,...,D,T,D,T,D,D,D,D,T,T
2,10_240913_C_G,10,240913,C,G,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,C258W,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,D,T,D,D,D,D,D,T
3,10_242031_A_C,10,242031,A,C,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,H281P,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,D,D,D,D,D,D,D,D
4,10_242115_G_A,10,242115,G,A,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R309H,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,D,D,D,D,D,D,D,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4484,Y_634761_T_G,Y,634761,T,G,1,NP_006874.1,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,Y141D,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4485,Y_634771_C_A,Y,634771,C,A,1,NP_006874.1,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,A144D,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4486,Y_634780_G_A,Y,634780,G,A,1,NP_006874.1,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,R147H,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4487,Y_640851_C_T,Y,640851,C,T,1,NP_006874.1,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,R173C,MEELTAFVSKSFDQKSKDGNGGGGGGGGKKDSITYREVLESGLARS...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Giả sử df là DataFrame hiện tại của bạn
print(f"🔹 Số lượng biến thể ban đầu: {len(final_df):,}")

# 1. Tự động nhận diện tất cả các cột điểm số 
# (Bao gồm: score, rankscore, phred, raw, và các điểm bảo tồn tiến hóa như gerp, phylop, phastcons)
score_cols = [
    col for col in final_df.columns 
    if any(keyword in col.lower() for keyword in ['score', 'phred', 'raw', 'gerp', 'phylop', 'phastcons'])
]

print(f"🔹 Đã tìm thấy {len(score_cols)} cột điểm số để kiểm tra.")

# 2. Xóa các dòng bị thiếu (NaN) ở BẤT KỲ cột điểm số nào trong danh sách trên
df_clean = final_df.dropna(subset=score_cols).copy()
df_clean[['AF', 'gnomADe_AF']] = df_clean[['AF', 'gnomADe_AF']].fillna(0)

# 3. Báo cáo kết quả
dropped_count = len(final_df) - len(df_clean)
print(f"🔹 Số lượng biến thể sau khi xóa: {len(df_clean):,}")
print(f"⚠️ Đã xóa tổng cộng: {dropped_count:,} dòng (chiếm {(dropped_count/len(final_df)*100):.2f}%)")

print(df_clean.isnull().sum()[10:70])
print(df_clean.isnull().sum()[70:])

df_clean.to_parquet(r"D:\variant_data\proteingym_after_vep_final.parquet", index=False)

df_clean

🔹 Số lượng biến thể ban đầu: 3,717
🔹 Đã tìm thấy 66 cột điểm số để kiểm tra.
🔹 Số lượng biến thể sau khi xóa: 1,472
⚠️ Đã xóa tổng cộng: 2,245 dòng (chiếm 60.40%)
Location                         0
Allele                        1472
Gene                             0
SYMBOL                           0
Feature                          0
Feature_type                     0
Consequence                      0
cDNA_position                    0
CDS_position                     0
Protein_position                 0
Amino_acids                      0
Codons                           0
MANE_SELECT                      0
CANONICAL                        0
AF                               0
gnomADe_AF                       0
phyloP100way_vertebrate          0
phyloP470way_mammalian           0
phyloP17way_primate              0
phastCons100way_vertebrate       0
phastCons470way_mammalian        0
phastCons17way_primate           0
GERP++_RS                        0
GERP++_NR                       

,Variant_ID,CHROM,POS,REF,ALT,Label,protein,protein_sequence,mutant,mutated_sequence,...,PHACTboost_pred,MutFormer_pred,MVP_pred,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred
1,10_180088_C_T,10,180088,C,T,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R26W,MARLTKRRQADTKAIQHLWAAIEIIWNQKQIANIDRITKYMSRVHG...,...,D,T,D,T,D,D,D,D,T,T
2,10_240913_C_G,10,240913,C,G,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,C258W,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,D,T,D,D,D,D,D,T
3,10_242031_A_C,10,242031,A,C,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,H281P,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,D,D,D,D,D,D,D,D
9,10_248502_G_A,10,248502,G,A,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S465N,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,T,T,T,T,T,D,T,T,T,T
10,10_248559_C_T,10,248559,C,T,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S484L,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,T,T,T,T,D,D,T,T,D,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4477,20_64207869_G_A,20,64207869,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,V225I,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,T,T,T,T,T
4478,20_64207986_G_C,20,64207986,G,C,1,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E264Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,D,T,T,T,T,D,T,T,T,T
4479,20_64208061_G_A,20,64208061,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E289K,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,D,T,T,T,T,T
4480,20_64208272_G_A,20,64208272,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,R359Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,T,T,T,T,T
